In [1]:
import numpy as np
import pandas as pd
import statsmodels as sm
import matplotlib.pyplot as plt
import scipy as sc


function for vec operator

In [2]:
def vec(mA):
    return np.asmatrix(mA.ravel('F'))


Function for time-series or cross-sectional de-meaning

In [3]:
def Demean(mA, iAxis):
    
        if iAxis == 0:
        
            mA = mA - np.mean(mA,axis=iAxis).reshape(1,int(np.shape(mA)[1-iAxis]))
        else:
            mA = mA - np.mean(mA,axis=iAxis).reshape(int(np.shape(mA)[1-iAxis]),1)
        
        return mA

Test

In [4]:
A = np.array([[1,2],[3,6],[5,7]])
vec(A)


matrix([[1, 3, 5, 2, 6, 7]])

In [5]:
Demean(A,0), Demean(A,1)


(array([[-2., -3.],
        [ 0.,  1.],
        [ 2.,  2.]]),
 array([[-0.5,  0.5],
        [-1.5,  1.5],
        [-1. ,  1. ]]))

Below is the function that can be used to generate the data

In [22]:
def GenrData (iSizeT, iSizeN, dAlpha, iS):
 
    mErrors  = np.random.randn(iSizeN, iSizeT+iS+1)
    mDataY = np.zeros((iSizeN,iSizeT+iS+1))

    for t in range(1,iSizeT+iS+1):    
    
        mDataY[:,[t]] = dAlpha*mDataY[:,[t-1]] +  mErrors[:,[t]]
    
 

    return (np.transpose(mDataY[:,iS+1:]), np.transpose(mDataY[:,iS:iSizeT+iS]))

In [23]:
#test this function
mDataY,mDataY_lag = GenrData (5, 100, 0.5,1)

In [21]:
print(np.shape(mDataY))

(5, 100)


This function provides FE-HPJ estimator, assuming that function that gives you FE estimator as inpute takes Y, X data only

In [19]:
def HPJ (mDataY, mDataX, dEstimator, Estimation):
    
    iSizeT,iSizeN = np.shape(mDataY)
    iHalf = int(np.floor(iSizeT/2))
    

    dEstimator1 = Estimation (mDataY[:iHalf,:], mDataX[:iHalf,:])
    
    dEstimator2 = Estimation (mDataY[iHalf:,:], mDataX[iHalf:,:])
    
    dHPJ = 2*dEstimator-0.5*(dEstimator1+dEstimator2)
    
    
    return (dHPJ,dEstimator1,dEstimator2) 

The following skeleton of bootstrap can be used to generate RDWB CI for both FE and HPJ (ising any type residuals). Output is a $[P\times 2]$ vector of indicator variables whether corresponding $H_{0}$ is rejected or not. $P$ can be a general number if you want to consider testing many $H_{0}$ at the same time, i.e. construct confidence intervals (parts 2 and 3 in Section 3).

In [ ]:
def BootstrapMC (mResiduals, vY0, iB, vNull, dLevel,vEst,dAlpha_O_bootstrap):
    
    iSizeT, iSizeN = np.shape(mResiduals)
    iSizeP = np.shape(vNull)[0]
    iSizeEst = int(np.shape(vEst)[0])
    
    iUnitLow = int(np.floor((dLevel/2)*iB))
    iUnitHigh = int(np.floor((1-dLevel/2)*iB))
    
    mResB = np.empty((iB,iSizeEst))
    mResB[0,:] = vEst.T
    
    for b in range(1,iB):
        
        #this should give Rademacher weights. Double check.
        mWeights = 
        
        mErrors_B = np.multiply((mResiduals,mWeights))
        
        # modife GenrData function to take as inputs initial condition,and matrix of residuals for error terms
        mDataY_B, mDataY_lag_B = GenrDataBootstrap (iSizeT, iSizeN, dAlpha_O_bootstrap,iS,vY0,mErrors_B)
        
        dFE_B = FE (mDataY_B, mDataY_lag_B)
        dFE_HPJ_B = HPJ (mDataY_B, mDataY_lag_B,dFE_B,FE)
        
        mResB[b,0] = dFE_B
        mResB[b,1] = dFE_HPJ_B 
    
    mReject = np.zeros((iSizeEst,iSizeP))
    mResB_Sorted = np.sort(mResB, axis=0)
    for k in range(0,iSizeEst):
               
        dLowQ = mResB_Sorted[iUnitLow,k]
        dHighQ = mResB_Sorted[iUnitHigh,k] 
        dLow = dLowQ
        dHigh = dHighQ
    
        mReject[k,:] = np.where((vNull < dHigh) & (vNull> dLow),0,1)
                   
    return mReject.T

Below I summarize the skeleton for the function that performs Monte Carlo study

In [18]:
def MC_study (iSizeT, dKappa,dGamma, iM,iB):
    
    
    # here you store your estimation results
    mResultsEstimation = np.zeros((iM,2))
    # here you store your testing results for any choice of P. Standard P=1
    iP = 1
    mResultsTesting = np.zeros((iM,2*iP)) 
    
    
    for m in range (0,iM):
        
        # Remember that the true value is a function of T,\gamma,\kappa
        dAlpha_0 = 
        
        iSizeN = 
        
        mDataY, mDataY_lag = GenrData (iSizeT, iSizeN, dAlpha_0,iS)
        
        dFE = FE (mDataY, mDataY_lag)
        dFE_HPJ = HPJ (mDataY, mDataY_lag,dFE,FE)
        
        
        mResiduals_FE = 
        
        #make sure that your residuals sum up to zero! This is not automatically satisfied for the HPJ method
        mResiduals_HPJ = 
        
        vEst = np.vstack((dFE,dFE_HPJ))
        
        #this can be either FE or HPJ, or true value of alpha_{0} that you used to generate the true DGPJ
        dAlpha_O_bootstrap = 
        
        #this can be either residuals from FE or HPJ estimation
        mResiduals_B = 
        vNull = dAlpha_0
        mReject = BootstrapMC (mResiduals_B, vY0, iB, vNull, dLevel,vEst,dAlpha_O_bootstrap)
        

        # it is convenient to store estimates in deviations from the true values
        mResultsEstimation[m,0] = dFE - dAlpha_0
        mResultsEstimation[m,1] = dHPJ - dAlpha_0
        
    #bias 
    vBias = 
    
    #RMSE 
    vRMSE = np.sqrt(np.mean(np.power(mResultsEstimation,2),axis=0))
    
    

    

SyntaxError: invalid syntax (<ipython-input-18-4cccaad7f0bd>, line 12)